# Task 6 · round 2: co-read features (plan_t6.md)

Round 1 (`task6_improve.ipynb`): `coread` −0.93% on A, `item_gaps` nothing, `n_steps` > 1 hurts,
bagging −0.2%. Shipped baseline: **A 7,182,796 · B 7,191,137**.

This round, all with `n_steps=1` and without `item_gaps`:
1. `coread` + `n_bag=5`: the round-1 winner without the harmful `n_steps=2`.
2. `coread_k` = 3, 5, 10: pool the co-read delay over the user's last k books (round 1 had
   the delay for only 39% of rows). Implied gap per co-reader v of history book a:
   (t_ua + t_vb − t_va) − t_last.

Cost: the co-read loop scales with k (k=10 may take several minutes per fit).

In [1]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import pandas as pd

from assignment_1.modules.models import GapModel, GapRegressor, split_last
from assignment_1.modules.util import load_csv

DATA_DIR = "../data"  # notebook runs from notebooks/
train = load_csv("task_6_train.csv", DATA_DIR)
trA, valA = split_last(train)
trB, valB = split_last(trA)
SHIPPED = {"A": 7_182_796, "B": 7_191_137}   # E1 of task6_improve.ipynb

In [2]:
results = {}

def run(name: str, backtests: str = "A", **kw) -> pd.DataFrame:
    row = results.setdefault(name, {})
    for b in backtests:
        tr, val = (trA, valA) if b == "A" else (trB, valB)
        t0 = time.time()
        pred = GapRegressor(base=GapModel(k=10, lam=1.5), **kw).fit(tr).predict(val)
        row[f"mae_{b}"] = float(np.mean(np.abs(pred - val["target"].to_numpy())))
        row[f"Δ%_{b}"] = 100 * (row[f"mae_{b}"] / SHIPPED[b] - 1)
        row[f"sec_{b}"] = round(time.time() - t0)
    return pd.DataFrame(results).T.round(2)

## R1. Coverage of the delay features by k (cheap, no model fit)

In [3]:
from assignment_1.modules.models.time.gaps import _coread_features

cov = {}
for k in (1, 3, 5, 10):
    t0 = time.time()
    f = _coread_features(trA, valA, k=k)
    cov[k] = {**f.notna().mean().round(3).to_dict(), "sec": round(time.time() - t0)}
pd.DataFrame(cov).T

,co_n,co_frac_after,co_q25,co_med,co_q75,sec
1,0.933,0.468,0.392,0.392,0.392,16.0
3,0.933,0.661,0.572,0.572,0.572,20.0
5,0.933,0.715,0.625,0.625,0.625,23.0
10,0.933,0.754,0.666,0.666,0.666,26.0


## R2. Round-1 winner without n_steps=2 (expect about −1.0% on A)

In [4]:
run("coread", coread=True)

,mae_A,Δ%_A,sec_A
coread,7115770.4,-0.93,50.0


In [5]:
run("coread, bag 5", coread=True, n_bag=5)

,mae_A,Δ%_A,sec_A
coread,7115770.40,-0.93,50.0
"coread, bag 5",7142050.56,-0.57,57.0


## R3. Co-read delay over the last k books

In [6]:
for k in (3, 5):
    table = run(f"coread k={k}", coread=True, coread_k=k)
table

,mae_A,Δ%_A,sec_A
coread,7115770.40,-0.93,50.0
"coread, bag 5",7142050.56,-0.57,57.0
coread k=3,7119102.54,-0.89,57.0
coread k=5,7104178.50,-1.09,62.0


In [7]:
run("coread k=10", coread=True, coread_k=10)   # slow; skip if k=5 did not beat k=3

,mae_A,Δ%_A,sec_A
coread,7115770.40,-0.93,50.0
"coread, bag 5",7142050.56,-0.57,57.0
coread k=3,7119102.54,-0.89,57.0
coread k=5,7104178.50,-1.09,62.0
coread k=10,7109593.36,-1.02,69.0


## R4. Best k + bagging, on both backtests

In [8]:
BEST = dict(coread=True, coread_k=3, n_bag=5)   # <- set coread_k from R3
run("best", "AB", **BEST)

,mae_A,Δ%_A,sec_A,mae_B,Δ%_B,sec_B
coread,7115770.40,-0.93,50.0,NaN,NaN,NaN
"coread, bag 5",7142050.56,-0.57,57.0,NaN,NaN,NaN
coread k=3,7119102.54,-0.89,57.0,NaN,NaN,NaN
coread k=5,7104178.50,-1.09,62.0,NaN,NaN,NaN
coread k=10,7109593.36,-1.02,69.0,NaN,NaN,NaN
best,7110401.53,-1.01,67.0,7096961.19,-1.31,62.0


## Decision
- Ship `BEST` if it beats the shipped model on both A and B: set `TASK6_REGRESSOR_PARAMS = BEST`
  in `tasks/task6.py`, then `make check`, `make changes`, upload.
- Earlier Task 6 gains carried over to public at about the same relative size, so −X% on A
  ≈ −X% on public. 8.5M (4.0) needs about −3%.